# Reproducing Mao et al. 2010

The second-order structure function of rotation measure (RM) is given by:

$$ SF_{\text{RM},\text{obs}}(\delta\theta) = \langle[\text{RM}(\theta) - \text{RM}(\theta+\delta\theta)]^2\rangle$$

That is, the ensemble average of the squared-difference in RM for sources with angular separation $\delta\theta$. We also need to correct for the impact of errors by:

$$ SF_{\text{RM}}(\delta\theta) = SF_{\text{RM},\text{obs}}(\delta\theta) - SF_{\sigma_\text{RM}}(\delta\theta) $$

See Haverkorn et al. 2004 (2004ApJ...609..776H) for details.

Here we'll reproduce the results of Mao et al. 2010 (2010ApJ...714.1170M).

In [ ]:
from importlib import resources

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import SkyCoord
from astropy.table import Table

from structurefunction import FitModel, StencilOptions, structure_function
from structurefunction.fitting import compare_models, evaluate, fit_sf
from structurefunction.io import to_table
from structurefunction.plotting import plot_sf

First, we can look at the expected results. I've pulled these data from the paper's plots using a web plot digitiser.

In [ ]:
mao_sf = (
    10
    ** np.array(
        [
            2.417894195735458,
            2.097768739428083,
            2.2926445508823967,
            2.164149088774578,
            2.1404402085381147,
            2.176621779798478,
            2.218782670913379,
            2.2444782900152482,
            2.2476963207124476,
            2.2837806390213578,
        ]
    )
    * (u.rad / u.m**2) ** 2
)
mao_sep = (
    10
    ** np.array(
        [
            -0.7729091483767441,
            -0.5386163683663935,
            -0.2730532911440767,
            -0.02550632317850443,
            0.21819567988496358,
            0.47213008276920787,
            0.7173429798998987,
            0.9643533199726302,
            1.18882007856649,
            1.3453070240944185,
        ]
    )
    * u.deg
)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(mao_sep, mao_sf, "X", label="Paper SF")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(10**1.5, 10**3)
ax.set_xlim(10**-1, 10**1.5)
ax.legend()
ax.set_xlabel(rf"$\Delta\theta$ [{mao_sep.unit:latex_inline}]")
ax.set_ylabel(rf"RM SF [{mao_sf.unit:latex_inline}]");

Now we'll load the original data (Table 1 of the paper, shipped with the package) and keep the sources Mao et al. used.

In [ ]:
with resources.as_file(
    resources.files("structurefunction.data") / "mao2010_ngp.ecsv"
) as path:
    mao_rm_tab = Table.read(path)

included = mao_rm_tab[mao_rm_tab["used_s5_2"]]
coords = SkyCoord(included["ra"], included["dec"])
rm = included["rm"].quantity
e_rm = included["rm_err"].quantity
mao_rm_tab

Now we'll define our own bin edges to compute a structure function.

In [ ]:
start = -0.8
stop = 1.5
step = 0.1
bins = 10 ** np.arange(start, stop + step, step) * u.deg

Here we compute the structure function itself, including the error correction. Uncertainties come from 70 jackknife regions: the sources are split into compact areas of sky of similar size, and the structure function is recomputed with each area left out in turn. The errors therefore include the spread between different parts of the field as well as the RM errors.

In [ ]:
sf_result = structure_function(rm, e_rm, coords, bins=bins, regions=70, rng=1)
to_table(sf_result)["separation", "sf", "sf_err", "count"]

We can compare our results to those from the original paper. We also plot our structure function before the error correction, $SF_{\text{RM},\text{obs}}$.

In [ ]:
sf_obs = sf_result.sf + sf_result.noise_bias

fig, ax = plt.subplots(figsize=(6, 6))
plot_sf(sf_result, ax=ax)
ax.plot(sf_result.separation, sf_obs, "s", mfc="none", label="Before error correction")
ax.plot(mao_sep, mao_sf, "X", label="Paper SF")
ax.set_ylim(10**1.5, 10**3)
ax.set_xlim(10**-1, 10**1.5)
ax.legend();

The paper's points follow our structure function before the error correction. After subtracting the errors, our structure function is lower by the mean of $\sigma_1^2 + \sigma_2^2$ over pairs, about 95 rad$^2$ m$^{-4}$ here.

In [ ]:
good = sf_result.count >= 10
interp_obs = np.interp(
    np.log10(mao_sep.value),
    np.log10(sf_result.separation.value[good]),
    np.log10(sf_obs[good]),
)
print(np.round(10**interp_obs / mao_sf.value, 2))
assert np.all(np.abs(interp_obs - np.log10(mao_sf.value)) < 0.15)
print(f"Mean error correction: {np.nanmean(sf_result.noise_bias):.1f}")

## Choosing a model

We fit four models to the same bins, using the full covariance between bins: a constant (no structure on these scales), a power law, a broken power law and a smoothly broken power law. `compare_models` ranks them by BIC and, with `sampler="nautilus"`, also gives the Bayesian evidence $\ln Z$.

A difference in BIC of more than about 6, or in $\ln Z$ of more than about 3, is strong support for the better model. Smaller differences mean the data can't really tell the models apart.

In [ ]:
table, fits = compare_models(sf_result, sampler="nautilus", rng=2)
for name in table.colnames[2:]:
    table[name].info.format = ".2f"
table

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
plot_sf(sf_result, ax=ax)
grid = np.geomspace(
    np.nanmin(sf_result.separation.value), np.nanmax(sf_result.separation.value), 200
)
for model, fit in fits.items():
    ax.plot(grid, evaluate(fit, grid), label=str(model))
ax.legend();

We take the model with the highest evidence and sample its posterior.

In [ ]:
best = FitModel(table["model"][np.argmax(table["log_evidence"])])
fit = fit_sf(sf_result, model=best, sampler="emcee", rng=3)
print(best)
for name, lo, med, hi in zip(fit.names, fit.lower, fit.median, fit.upper, strict=True):
    print(f"{name}: {med:.2f} (+{hi - med:.2f} / -{med - lo:.2f})")
fig, ax = plt.subplots(figsize=(6, 6))
plot_sf(sf_result, fit=fit, ax=ax, rng=4);

Finally, we extend to multi-point structure functions, as described by Seta et al. 2022 (10.1093/mnras/stac2972). The three-point structure function is

$$ SF^{(3)}_{\text{RM}}(\delta\theta) = \langle[\text{RM}(\theta+\delta\theta) - 2\,\text{RM}(\theta) + \text{RM}(\theta-\delta\theta)]^2\rangle $$

On irregularly spaced sources we accept a triplet when the outer two sources are within `angle_tol_deg` of a straight line through the middle source, and their distances from it agree to within `sep_tol`.

In [ ]:
sf3_result = structure_function(
    rm,
    e_rm,
    coords,
    n_point=3,
    bins=bins,
    regions=70,
    stencil=StencilOptions(angle_tol_deg=15.0, sep_tol=0.2, mode="exact"),
    rng=1,
)
fig, ax = plt.subplots(figsize=(6, 6))
plot_sf(sf3_result, ax=ax);